# AdamW / Muon / MuonClip-RMS fixed-point comparison

This notebook reads complete, matched three-seed arms. It tests the
baseline premise--high accuracy together with credible late
`alpha ~= 2` and independently supported trace-log near zero--before
any transformed tangent operator is interpreted.


In [ ]:
# Papermill parameters. Override these values in an injected cell.
RUN_ROOT = ""
OUTPUT_ROOT = ""
CHECKPOINT_CACHE_ROOT = ""
CONFIG_PATH = ""
PROFILE = "pilot_1000_epochs"
PROTOCOL_SLUG = ""
SEEDS = [1337, 2027, 31415]
CHECKPOINT_PAYLOAD_CACHE_SIZE = 24
SHOW_PLOTS = True
REQUIRE_ARTIFACTS = True
ALLOW_TEMPORARY_LONG_RUN = False
OPTIMIZER_SLUGS = ["adamw", "muon", "muonclip_rms"]
ALPHA_TOLERANCE = 0.25
TRACE_LOG_PER_EVAL_TOLERANCE = 0.10
MINIMUM_TAIL = 8
MAXIMUM_KS_D = 0.15


In [ ]:
from pathlib import Path
from dataclasses import asdict, is_dataclass
from functools import lru_cache
import inspect
import json
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "baseline" / "rg_baselines").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find baseline/rg_baselines. Launch Jupyter from a clone of "
        "CalculatedContent/rg_optimizers."
    )
BASELINE_ROOT = REPO_ROOT / "baseline"
EXPERIMENT_ROOT = BASELINE_ROOT / "experiments" / "mnist_mlp3_tangent_rg"
if str(BASELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(BASELINE_ROOT))

default_root = os.environ.get(
    "RG_MNIST_TANGENT_ROOT", "/tmp/rg-mnist-mlp3-tangent-rg"
)
RUN_ROOT_PATH = Path(RUN_ROOT or default_root).expanduser().resolve()

default_checkpoint_cache_root = os.environ.get(
    "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT",
    "/tmp/rg-mnist-mlp3-tangent-checkpoints",
)
CHECKPOINT_CACHE_ROOT_PATH = Path(
    CHECKPOINT_CACHE_ROOT or default_checkpoint_cache_root
).expanduser().resolve()

def _suite_name_from_profile():
    if str(PROTOCOL_SLUG).strip():
        return str(PROTOCOL_SLUG).strip()
    candidate = (
        Path(CONFIG_PATH).expanduser()
        if str(CONFIG_PATH).strip()
        else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
    )
    if candidate.is_file():
        if candidate.suffix.lower() == ".json":
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            value = payload.get("protocol", {}).get("suite_name")
            if value:
                return str(value)
        else:
            for line in candidate.read_text(encoding="utf-8").splitlines():
                stripped = line.strip()
                if stripped.startswith("suite_name:"):
                    return stripped.split(":", 1)[1].strip().strip("'\"")
    fallback = {
        "smoke": "mnist_mlp3_tangent_rg_v1_smoke",
        "pilot_1000_epochs": "mnist_mlp3_tangent_rg_v1_pilot1000",
        "long_horizon_10000_epochs": "mnist_mlp3_tangent_rg_v1_reference10000",
    }
    if PROFILE not in fallback:
        raise FileNotFoundError(
            f"Cannot derive suite_name for PROFILE={PROFILE!r}; set CONFIG_PATH "
            "or PROTOCOL_SLUG explicitly."
        )
    return fallback[PROFILE]

PROTOCOL_SLUG = _suite_name_from_profile()
OUTPUT_ROOT_PATH = Path(
    OUTPUT_ROOT or RUN_ROOT_PATH / PROTOCOL_SLUG / "notebook_outputs"
).expanduser().resolve()
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)

SEEDS = tuple(int(seed) for seed in SEEDS)
if SEEDS != (1337, 2027, 31415):
    print("WARNING: this is not the preregistered three-seed tuple:", SEEDS)

print("repository:", REPO_ROOT)
print("run root:", RUN_ROOT_PATH)
print("tail checkpoint cache root:", CHECKPOINT_CACHE_ROOT_PATH)
print("effective suite:", PROTOCOL_SLUG)
print("output root:", OUTPUT_ROOT_PATH)
print("seeds:", SEEDS)

EXPERIMENT_ROOT = BASELINE_ROOT / 'experiments' / 'mnist_mlp3_tangent_rg'


In [ ]:
from rg_baselines.statistics import summarize_numeric_metrics
from rg_baselines.tangent_rg import powerlaw_fit, trace_log


import subprocess

from rg_baselines.tangent_rg import (
    AdamWProfile,
    MuonClipRMSProfile,
    MuonProfile,
    TangentRGConfig,
    build_analysis_plan,
    list_analysis_checkpoints,
    list_capture_files,
    load_config,
    run_training,
)
from rg_baselines.tangent_rg.checkpoints import load_verified_tail_checkpoint_refs
from rg_baselines.tangent_rg.protocol import tail_checkpoint_epochs
from rg_baselines.tangent_rg import cli as tangent_cli

from rg_baselines.tangent_rg import plotting, reporting


## Baseline observable

**`operator_kind`: `raw_weight_gram_esd_fixed_point_comparison`**

**`map_definition`: `Matched optimizer comparison of raw layer Gram ESD fits and independently supported trace-log rows.`**

**Identifiability caveat.** Agreement with alpha=2 does not identify a quotient or tangent map. This notebook qualifies the baseline regime only; it cannot select a transformed operator.

These strings are persisted with every result row. A visually useful
spectrum does not change the identity of the map that produced it.


In [ ]:
def require_path(path, *, description="artifact"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "Run the prerequisite numbered notebook or set RUN_ROOT / "
            "OUTPUT_ROOT to the completed protocol directory."
        )
    return path


def first_existing(directory, names, *, description):
    directory = Path(directory)
    candidates = [directory / name for name in names]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {description} beneath {directory}. Expected one of:\n"
        + "\n".join(f"  - {path}" for path in candidates)
    )


def resolve_protocol_root():
    direct = RUN_ROOT_PATH / PROTOCOL_SLUG
    return direct if direct.is_dir() else RUN_ROOT_PATH


def resolve_arm_dir(optimizer_slug):
    protocol = resolve_protocol_root()
    candidates = [
        protocol / optimizer_slug,
        protocol / "results" / optimizer_slug,
        RUN_ROOT_PATH / optimizer_slug,
        RUN_ROOT_PATH / "results" / optimizer_slug,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    if REQUIRE_ARTIFACTS:
        raise FileNotFoundError(
            f"No completed {optimizer_slug!r} arm was found. Checked:\n"
            + "\n".join(f"  - {path}" for path in candidates)
        )
    return candidates[0]


def resolve_seed_dir(optimizer_slug, seed):
    arm = resolve_arm_dir(optimizer_slug)
    candidates = [
        arm / f"seed_{int(seed)}",
        arm / f"seed_{int(seed):05d}",
        arm / "seeds" / f"seed_{int(seed)}",
        arm / "seeds" / f"seed_{int(seed):05d}",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Missing seed directory for optimizer={optimizer_slug}, seed={seed}. "
        f"Checked {candidates}."
    )


def validate_run_identity(seed_dir, *, optimizer_slug, seed):
    seed_dir = Path(seed_dir)
    manifest = json.loads(
        require_path(seed_dir / "manifest.json", description="run manifest")
        .read_text(encoding="utf-8")
    )
    resolved = json.loads(
        require_path(seed_dir / "resolved_config.json", description="resolved config")
        .read_text(encoding="utf-8")
    )
    completion = json.loads(
        require_path(seed_dir / "run_complete.json", description="completion marker")
        .read_text(encoding="utf-8")
    )
    config = dict(resolved.get("config", resolved))
    checks = {
        "manifest suite": (manifest.get("suite_name"), PROTOCOL_SLUG),
        "resolved suite": (config.get("suite_name"), PROTOCOL_SLUG),
        "manifest optimizer": (manifest.get("optimizer"), optimizer_slug),
        "resolved optimizer": (config.get("optimizer"), optimizer_slug),
        "completion optimizer": (completion.get("optimizer"), optimizer_slug),
        "manifest seed": (manifest.get("seed"), int(seed)),
        "resolved seed": (config.get("seed"), int(seed)),
        "completion seed": (completion.get("seed"), int(seed)),
    }
    mismatches = [
        f"{label}: observed={observed!r}, expected={expected!r}"
        for label, (observed, expected) in checks.items()
        if str(observed) != str(expected)
    ]
    fingerprints = {
        str(manifest.get("protocol_fingerprint", "")),
        str(resolved.get("protocol_fingerprint", "")),
        str(completion.get("protocol_fingerprint", "")),
    }
    if "" in fingerprints or len(fingerprints) != 1:
        mismatches.append(
            "manifest/resolved/completion protocol fingerprints are missing or unequal"
        )
    if not bool(completion.get("completed", False)):
        mismatches.append("run_complete.json does not declare completed=true")
    try:
        resolved_epochs = int(config["epochs"])
        completion_epochs = int(completion["epochs"])
        completion_step = int(completion["global_step"])
        best_validation_epoch = int(completion["best_validation_epoch"])
        analysis_plan = dict(resolved["analysis_plan"])
        plan_steps_per_epoch = int(analysis_plan["steps_per_epoch"])
        plan_total_steps = int(analysis_plan["total_steps"])
    except (KeyError, TypeError, ValueError) as error:
        mismatches.append(
            "resolved/completion final-horizon metadata is missing or invalid: "
            f"{type(error).__name__}: {error}"
        )
    else:
        if resolved_epochs < 1 or plan_steps_per_epoch < 1:
            mismatches.append("resolved epochs and steps_per_epoch must be positive")
        if plan_total_steps != resolved_epochs * plan_steps_per_epoch:
            mismatches.append(
                "resolved analysis_plan total_steps does not equal "
                "epochs * steps_per_epoch"
            )
        if completion_epochs != resolved_epochs:
            mismatches.append(
                f"completion epochs={completion_epochs} != resolved epochs={resolved_epochs}"
            )
        if completion_step != plan_total_steps:
            mismatches.append(
                f"completion global_step={completion_step} != resolved "
                f"analysis_plan total_steps={plan_total_steps}"
            )
        if not 0 <= best_validation_epoch <= resolved_epochs:
            mismatches.append(
                f"best_validation_epoch={best_validation_epoch} is outside "
                f"[0, {resolved_epochs}]"
            )
    if mismatches:
        raise RuntimeError(
            f"Run identity/provenance mismatch beneath {seed_dir}:\n  - "
            + "\n  - ".join(mismatches)
        )
    return manifest, resolved, completion


def validate_cross_run_provenance(manifests):
    manifests = list(manifests)
    if not manifests:
        raise RuntimeError("No manifests supplied for cross-run provenance audit")
    invariant_fields = (
        "suite_name", "dataset", "model", "initialization", "normalization",
        "train_indices_sha256", "validation_indices_sha256",
        "test_monitoring_only", "analysis_plan", "device", "software_versions",
        "determinism_settings",
    )
    disagreements = []
    for field in invariant_fields:
        serialized = {
            json.dumps(item.get(field), sort_keys=True, default=str)
            for item in manifests
        }
        if len(serialized) != 1:
            disagreements.append(field)
    if disagreements:
        raise RuntimeError(
            "Matched arms disagree on frozen run provenance fields: "
            + ", ".join(disagreements)
        )
    identities = {
        (str(item.get("optimizer")), int(item.get("seed"))) for item in manifests
    }
    expected = {
        (str(optimizer), int(seed))
        for optimizer in OPTIMIZER_SLUGS
        for seed in SEEDS
    } if "OPTIMIZER_SLUGS" in globals() else identities
    if identities != expected:
        raise RuntimeError(
            f"Manifest optimizer/seed grid is incomplete: observed={sorted(identities)}, "
            f"expected={sorted(expected)}"
        )
    return pd.DataFrame([
        {
            "optimizer": item.get("optimizer"),
            "seed": item.get("seed"),
            "device": item.get("device"),
            "software_versions": json.dumps(
                item.get("software_versions"), sort_keys=True, default=str
            ),
            "determinism_settings": json.dumps(
                item.get("determinism_settings"), sort_keys=True, default=str
            ),
            "pooling_compatibility_policy": (
                "headline pooling requires identical device, software versions, "
                "determinism settings, and scientific invariants across all runs"
            ),
        }
        for item in manifests
    ])


def record_dict(value):
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, dict):
        return dict(value)
    if hasattr(value, "__dict__"):
        return dict(vars(value))
    raise TypeError(f"Cannot convert {type(value).__name__} to an audit row")


def records_from_result(result):
    if result is None:
        return []
    if is_dataclass(result):
        return [record_dict(result)]
    if isinstance(result, dict):
        if "operator_kind" in result:
            return [dict(result)]
        rows = []
        for value in result.values():
            rows.extend(records_from_result(value))
        return rows
    if isinstance(result, (tuple, list)):
        rows = []
        for value in result:
            rows.extend(records_from_result(value))
        return rows
    return [record_dict(result)]


def spectrum_from_record(row):
    for name in (
        "spectrum", "eigenvalues", "singular_values", "rates",
        "positive_spectrum", "gram_spectrum",
    ):
        if name in row:
            values = np.asarray(row[name], dtype=float).reshape(-1)
            return values[np.isfinite(values) & (values > 0.0)]
    raise KeyError(
        "Operator record contains no recognized positive spectrum field. "
        f"Available fields: {sorted(row)}"
    )


def positive_spectrum(values, *, minimum_count=2):
    sample = np.asarray(values, dtype=float).reshape(-1)
    sample = sample[np.isfinite(sample) & (sample > 0.0)]
    sample = np.sort(sample)
    if sample.size < int(minimum_count):
        raise ValueError(
            f"Need at least {minimum_count} finite positive spectral values; "
            f"found {sample.size}."
        )
    return sample


def fit_spectrum_with_trace(
    values,
    *,
    operator_kind,
    map_definition,
    spectrum_kind,
    metadata,
    top_k_values=(0, 1, 2, 3, 4, 5),
    minimum_tail=8,
):
    # Fit amplitudes once, transform that fit to energy, and audit trace-log.
    # The power-law package is never called independently on squared values.
    # Trace-log uses squared values at the amplitude fit's independent rank.

    if str(spectrum_kind) != "amplitude":
        raise ValueError(
            "fit_spectrum_with_trace accepts operator amplitudes only; "
            "energy rows are produced by the exact amplitude-to-energy transform."
        )

    sample = positive_spectrum(values)
    feasible_top_k = tuple(
        int(value) for value in top_k_values if int(value) <= sample.size - 2
    )
    if not feasible_top_k or feasible_top_k[0] != 0:
        feasible_top_k = (0,)
    amplitude_fits = powerlaw_fit.fit_clipping_sensitivity(
        sample,
        top_k_values=feasible_top_k,
        minimum_tail=int(minimum_tail),
        operator_kind=str(operator_kind),
        map_definition=str(map_definition),
        spectrum_kind="amplitude",
        metadata=dict(metadata),
    )
    energy_rows = [
        powerlaw_fit.amplitude_fit_to_energy(row)
        for row in amplitude_fits.to_dict(orient="records")
    ]
    fits = pd.concat(
        [amplitude_fits, pd.DataFrame(energy_rows)],
        ignore_index=True,
        sort=False,
    )
    primary = amplitude_fits.loc[amplitude_fits["clip_top_k"].eq(0)].iloc[0]
    energy = sample ** 2
    trace_row = {
        **dict(metadata),
        "operator_kind": str(operator_kind),
        "map_definition": str(map_definition),
        "spectrum_kind": "energy_derived_from_amplitude",
        "support_rank_source": "powerlaw.Fit package-selected xmin tail count",
        "support_selected_from_same_trace_log": False,
        "support_rank": int(primary.get("n_tail", 0)),
        "trace_log_total": np.nan,
        "trace_log_per_eval": np.nan,
        "lambda_cut_scaled": np.nan,
        "trace_status": "fit_has_no_supported_tail",
    }
    rank = int(primary.get("n_tail", 0))
    if rank > 0:
        evaluated = trace_log.trace_log_at_rank(
            energy,
            rank=min(rank, energy.size),
            normalization_dimension=float(energy.size),
            rank_source="powerlaw.Fit package-selected xmin tail count",
        )
        trace_row.update(evaluated)
        trace_row["trace_status"] = "ok"
    return fits, pd.DataFrame([trace_row])


def save_analysis_frames(method_slug, *, operators, fits, traces):
    destination = OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
    destination.mkdir(parents=True, exist_ok=True)
    required_identity = {
        "optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"
    }
    for label, frame in (("operators", operators), ("fits", fits), ("traces", traces)):
        missing = required_identity - set(frame.columns)
        if missing:
            raise RuntimeError(
                f"{method_slug} {label} lack analysis provenance: {sorted(missing)}"
            )
        if frame[list(required_identity)].isna().any().any():
            raise RuntimeError(f"{method_slug} {label} contain null analysis provenance")
    identity_rows = fits[
        ["optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"]
    ].drop_duplicates()
    duplicate_fingerprints = (
        identity_rows.groupby(["optimizer", "seed"], dropna=False)[
            "protocol_fingerprint"
        ].nunique()
    )
    if (duplicate_fingerprints != 1).any():
        raise RuntimeError(
            f"{method_slug} has multiple protocol fingerprints for one optimizer/seed"
        )
    fingerprint_grid = {
        f"{row.optimizer}:{int(row.seed)}": str(row.protocol_fingerprint)
        for row in identity_rows.itertuples(index=False)
    }
    expected_grid_count = identity_rows[["optimizer", "seed"]].drop_duplicates().shape[0]
    if len(fingerprint_grid) != expected_grid_count:
        raise RuntimeError(f"{method_slug} fingerprint-grid keys are not unique")
    provenance_manifest = {
        "schema_version": 1,
        "suite_name": str(PROTOCOL_SLUG),
        "method_slug": str(method_slug),
        "optimizer_seed_protocol_fingerprints": dict(sorted(fingerprint_grid.items())),
        "source_artifact_kinds": sorted(
            identity_rows["source_artifact_kind"].astype(str).unique().tolist()
        ),
        "operator_row_count": int(len(operators)),
        "fit_row_count": int(len(fits)),
        "trace_row_count": int(len(traces)),
    }
    provenance_manifest["analysis_contract_tokens"] = sorted(
        fits["analysis_contract_token"].dropna().astype(str).unique().tolist()
        if "analysis_contract_token" in fits.columns
        else []
    )
    operators.to_csv(destination / "operator_rows.csv", index=False)
    fits.to_csv(destination / "powerlaw_fits.csv", index=False)
    traces.to_csv(destination / "trace_log_independent_support.csv", index=False)
    (destination / "method_provenance.json").write_text(
        json.dumps(provenance_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return destination


def save_spectrum_ccdf_gallery(
    spectral_arrays,
    *,
    method_slug,
    maximum_panels=24,
):
    # Save bounded log-log PDF/CCDF diagnostics for positive amplitudes.
    gallery = OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "spectrum_pdf_ccdf"
    gallery.mkdir(parents=True, exist_ok=True)
    rows = []
    for index, (key, raw) in enumerate(sorted(spectral_arrays.items())):
        if index >= int(maximum_panels):
            break
        sample = positive_spectrum(raw)
        x, ccdf = powerlaw_fit.empirical_ccdf(sample)
        fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))
        if sample[0] < sample[-1]:
            bins = np.geomspace(sample[0], sample[-1], min(50, max(8, sample.size // 3)))
            axes[0].hist(sample, bins=bins, density=True, histtype="step", linewidth=1.8)
        else:
            axes[0].scatter(sample, np.ones_like(sample), s=15)
        axes[1].step(x, ccdf, where="post", linewidth=1.8)
        for axis in axes:
            axis.set_xscale("log")
            axis.set_yscale("log")
            axis.grid(alpha=0.2)
        axes[0].set(xlabel="amplitude b", ylabel="density", title="PDF")
        axes[1].set(xlabel="amplitude b", ylabel="P(B >= b)", title="CCDF")
        fig.suptitle(str(key), fontsize=8)
        fig.tight_layout()
        safe = "".join(character if character.isalnum() or character in "-_" else "_" for character in str(key))
        path = gallery / f"{index:03d}_{safe[:160]}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        if SHOW_PLOTS and index < 3:
            plt.show()
        else:
            plt.close(fig)
        rows.append({"spectrum_key": str(key), "n_positive": int(sample.size), "figure": str(path)})
    index_frame = pd.DataFrame(rows)
    index_frame.to_csv(gallery / "index.csv", index=False)
    return index_frame


def plot_fit_alpha_ci(fits, *, method_slug, title):
    usable = fits.copy()
    if "fit_ok" in usable:
        usable = usable[boolean_series(usable["fit_ok"])]
    if "spectrum_kind" in usable:
        energy = usable[
            usable["spectrum_kind"].astype(str).eq("energy_derived_from_amplitude")
        ]
        if not energy.empty:
            usable = energy
    primary = usable[usable["clip_top_k"].eq(0)] if "clip_top_k" in usable else usable
    if primary.empty:
        raise RuntimeError(
            f"{method_slug}: no successful preregistered raw fits; inspect powerlaw_fits.csv"
        )
    if "state_index" not in primary:
        primary["state_index"] = 0
    groups = tuple(
        name
        for name in (
            "optimizer", "layer", "method", "null_kind", "pair_stride",
            "epsilon", "evidence_role",
        )
        if name in primary
    )
    if not groups:
        primary["method"] = str(method_slug)
        groups = ("method",)
    return plot_seed_ci(
        primary,
        x="state_index",
        metric="alpha",
        groups=groups,
        title=title,
        ylabel="Power-law density exponent alpha",
        reference=2.0,
        allow_incomplete=True,
        incomplete_output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
            / "incomplete_alpha_ci_groups.csv"
        ),
        output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "alpha_95ci.png"
        ),
    )


def call_supported(function, /, *args, **kwargs):
    signature = inspect.signature(function)
    if any(
        parameter.kind is inspect.Parameter.VAR_KEYWORD
        for parameter in signature.parameters.values()
    ):
        return function(*args, **kwargs)
    supported = {key: value for key, value in kwargs.items() if key in signature.parameters}
    return function(*args, **supported)


def boolean_series(values):
    if getattr(values, "dtype", None) == bool:
        return values
    return values.astype(str).str.strip().str.lower().isin({"1", "true", "yes"})


def ci_summary(
    frame,
    *,
    groups,
    metrics,
    allow_incomplete=False,
    incomplete_output_path=None,
    return_incomplete=False,
):
    missing = set((*groups, *metrics, "seed")) - set(frame.columns)
    if missing:
        raise ValueError(f"CI input is missing columns: {sorted(missing)}")
    # Repeated layers/checkpoints/probes are not independent replicates.  First
    # collapse every declared group to one value per complete training seed.
    replicate = (
        frame.groupby([*groups, "seed"], as_index=False, dropna=False)[list(metrics)]
        .mean(numeric_only=True)
    )
    summary = summarize_numeric_metrics(
        replicate,
        group_columns=tuple(groups),
        metrics=tuple(metrics),
        confidence=0.95,
    )
    if summary.empty:
        raise RuntimeError("Confidence-interval summary is empty after seed aggregation")
    incomplete = summary[pd.to_numeric(summary["n"], errors="coerce") != len(SEEDS)]
    if not incomplete.empty:
        if incomplete_output_path is not None:
            incomplete_output_path = Path(incomplete_output_path)
            incomplete_output_path.parent.mkdir(parents=True, exist_ok=True)
            incomplete.to_csv(incomplete_output_path, index=False)
        if allow_incomplete:
            print(
                "WARNING: dropping incomplete CI identities from the mean/band; "
                "faint individual-seed traces remain visible.\n"
                + incomplete[
                    [name for name in (*groups, "metric", "n") if name in incomplete]
                ].to_string(index=False)
            )
        else:
            identity = [name for name in (*groups, "metric", "n") if name in incomplete]
            raise RuntimeError(
                "Every confidence-interval row requires exactly the preregistered "
                f"{len(SEEDS)} complete seeds. Incomplete identities:\n"
                + incomplete[identity].to_string(index=False)
            )
    complete = summary[pd.to_numeric(summary["n"], errors="coerce") == len(SEEDS)].copy()
    if return_incomplete:
        return complete, incomplete.copy()
    return complete


def plot_seed_ci(
    frame,
    *,
    x,
    metric,
    groups,
    title,
    ylabel,
    reference=None,
    output_path=None,
    allow_incomplete=False,
    incomplete_output_path=None,
):
    groups = tuple(groups)
    if allow_incomplete and incomplete_output_path is None and output_path is not None:
        output_path_for_report = Path(output_path)
        incomplete_output_path = output_path_for_report.with_name(
            output_path_for_report.stem + "_incomplete_ci_groups.csv"
        )
    summary, incomplete = ci_summary(
        frame,
        groups=(*groups, x),
        metrics=(metric,),
        allow_incomplete=allow_incomplete,
        incomplete_output_path=incomplete_output_path,
        return_incomplete=True,
    )
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    if not groups:
        frame = frame.copy()
        frame["series"] = "all"
        groups = ("series",)
    for identity, group in frame.groupby(list(groups), dropna=False):
        identity = identity if isinstance(identity, tuple) else (identity,)
        label = ", ".join(f"{key}={value}" for key, value in zip(groups, identity))
        for _, seed_frame in group.groupby("seed"):
            ordered = (
                seed_frame.groupby(x, as_index=False, dropna=False)[metric]
                .mean(numeric_only=True)
                .sort_values(x)
            )
            ax.plot(ordered[x], ordered[metric], alpha=0.16, linewidth=0.9)
        selected = summary.copy()
        for key, value in zip(groups, identity):
            selected = selected[selected[key].astype(str) == str(value)]
        selected = selected[selected["metric"] == metric].sort_values(x)
        if selected.empty:
            pass
        else:
            xv = selected[x].to_numpy(dtype=float)
            mean = selected["mean"].to_numpy(dtype=float)
            low = selected["ci_low"].to_numpy(dtype=float)
            high = selected["ci_high"].to_numpy(dtype=float)
            ax.plot(xv, mean, marker="o", linewidth=2.1, label=label)
            finite = np.isfinite(low) & np.isfinite(high)
            ax.fill_between(xv[finite], low[finite], high[finite], alpha=0.18)
        missing = incomplete.copy()
        for key, value in zip(groups, identity):
            missing = missing[missing[key].astype(str) == str(value)]
        missing = missing[missing["metric"] == metric]
        if not missing.empty:
            ax.scatter(
                missing[x].to_numpy(dtype=float),
                missing["mean"].to_numpy(dtype=float),
                marker="x", color="#555555", alpha=0.65, zorder=4,
            )
    if reference is not None:
        ax.axhline(float(reference), color="#333333", linestyle="--", linewidth=1.4)
    ax.set(xlabel=x, ylabel=ylabel, title=title)
    ax.set_xscale("symlog", linthresh=1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    return summary, fig


def resolve_config_path(profile, explicit=""):
    if explicit:
        return require_path(explicit, description="training configuration")
    names = (f"{profile}.yaml", f"{profile}.yml", f"{profile}.json")
    roots = (
        EXPERIMENT_ROOT / "configs",
        BASELINE_ROOT / "rg_baselines" / "tangent_rg" / "configs",
        BASELINE_ROOT / "configs" / "mnist_mlp3_tangent_rg",
    )
    for root in roots:
        for name in names:
            candidate = root / name
            if candidate.is_file():
                return candidate
    epoch_by_profile = {
        "smoke": 2,
        "pilot_1000_epochs": 1_000,
        "long_horizon_10000_epochs": 10_000,
    }
    if profile not in epoch_by_profile:
        raise FileNotFoundError(
            f"No checked-in configuration for profile={profile!r}. Set CONFIG_PATH."
        )
    epochs = epoch_by_profile[profile]
    generated = OUTPUT_ROOT_PATH / "generated_configs" / f"{profile}.json"
    generated.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "protocol": {
            "suite_name": PROTOCOL_SLUG,
            "schema_version": 1,
            "description": f"Notebook-generated preregistered {profile} profile",
        },
        "training": {
            "epochs": epochs,
            "lr_schedule_epochs": min(30, epochs),
            "batch_size": 128,
            "validation_size": 5_000,
            "split_seed": 20_260_807,
            "validation_every_epochs": 1 if epochs == 2 else 5,
            "latest_every_epochs": 1,
            "test_monitoring_only": True,
        },
        "analysis": {
            "log_points": 2 if epochs == 2 else 96,
            "explicit_epochs": [0, 1, 2, 5, 10, 30],
            "dense_burst_anchor_epochs": [0, 1, 10, 100, 1_000],
            "dense_burst_length_steps": 4 if epochs == 2 else 8,
            "capture_parameter_names": ["fc1.weight", "fc2.weight"],
        },
        "runtime": {
            "device": "auto",
            "data_dir": str(RUN_ROOT_PATH / "data"),
            "run_root": str(RUN_ROOT_PATH),
            "tail_checkpoint_cache_root": str(CHECKPOINT_CACHE_ROOT_PATH),
        },
    }
    generated.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    checked = load_config(generated)
    checked.validate()
    print("generated validated config:", generated)
    return generated


def run_cli_training(*, optimizer, profile, seeds, resume):
    config_path = resolve_config_path(profile, CONFIG_PATH)
    launch_config = load_config(config_path)
    if str(launch_config.suite_name) != str(PROTOCOL_SLUG):
        raise RuntimeError(
            f"Resolved config suite_name={launch_config.suite_name!r} does not match "
            f"the notebook output suite {PROTOCOL_SLUG!r}. Clear PROTOCOL_SLUG or "
            "select the matching PROFILE/CONFIG_PATH."
        )
    temporary_root = Path("/tmp").resolve()
    if (
        int(launch_config.epochs) > 2
        and RUN_ROOT_PATH.is_relative_to(temporary_root)
        and not bool(ALLOW_TEMPORARY_LONG_RUN)
    ):
        raise RuntimeError(
            f"Refusing long-horizon training under temporary root {RUN_ROOT_PATH}. "
            "Set RUN_ROOT or RG_MNIST_TANGENT_ROOT to persistent storage. "
            "For a deliberate disposable run only, set "
            "ALLOW_TEMPORARY_LONG_RUN=True."
        )
    for seed in seeds:
        command = [
            sys.executable,
            "-m",
            "rg_baselines.tangent_rg.cli",
            "train",
            "--config",
            str(config_path),
            "--optimizer",
            str(optimizer),
            "--output-root",
            str(RUN_ROOT_PATH),
            "--tail-checkpoint-root",
            str(CHECKPOINT_CACHE_ROOT_PATH),
            "--seed",
            str(int(seed)),
        ]
        if resume:
            command.append("--resume")
        print("running:", " ".join(command))
        subprocess.run(command, cwd=BASELINE_ROOT, check=True)


def load_arm_outputs(optimizer):
    arm = resolve_arm_dir(optimizer)
    manifests = []
    performance_frames = []
    spectral_frames = []
    for seed in SEEDS:
        seed_dir = resolve_seed_dir(optimizer, seed)
        manifest, resolved, completion = validate_run_identity(
            seed_dir, optimizer_slug=optimizer, seed=seed
        )
        manifests.append(manifest)
        performance_path = first_existing(
            seed_dir / "metrics",
            (
                "performance_by_analysis_epoch.csv",
                "performance_by_analysis_state.csv",
            ),
            description="analysis-state performance table",
        )
        performance = pd.read_csv(performance_path)
        for column, expected in (("optimizer", optimizer), ("seed", int(seed))):
            if column not in performance:
                raise KeyError(f"{performance_path} lacks required identity column {column}")
            observed = set(performance[column].dropna().astype(str))
            if observed != {str(expected)}:
                raise RuntimeError(
                    f"{performance_path} identity mismatch for {column}: "
                    f"observed={sorted(observed)}, expected={expected}"
                )
        if "protocol_fingerprint" not in performance:
            raise KeyError(f"{performance_path} lacks protocol_fingerprint")
        performance_fingerprints = set(
            performance["protocol_fingerprint"].dropna().astype(str)
        )
        if performance_fingerprints != {str(manifest["protocol_fingerprint"])}:
            raise RuntimeError(
                f"{performance_path} fingerprint mismatch: "
                f"observed={sorted(performance_fingerprints)}"
            )
        performance["seed"] = int(seed)
        performance["optimizer"] = optimizer
        performance_frames.append(performance)
        spectral_path = first_existing(
            seed_dir / "metrics",
            (
                "spectral_metrics_by_analysis_epoch.csv",
                "weightwatcher_fits.csv",
                "weightwatcher_by_analysis_epoch.csv",
            ),
            description="raw + clip_xmax spectral fit table",
        )
        spectral = pd.read_csv(spectral_path)
        for column, expected in (("optimizer", optimizer), ("seed", int(seed))):
            if column not in spectral:
                raise KeyError(f"{spectral_path} lacks required identity column {column}")
            observed = set(spectral[column].dropna().astype(str))
            if observed != {str(expected)}:
                raise RuntimeError(
                    f"{spectral_path} identity mismatch for {column}: "
                    f"observed={sorted(observed)}, expected={expected}"
                )
        if "protocol_fingerprint" not in spectral:
            raise KeyError(f"{spectral_path} lacks protocol_fingerprint")
        spectral_fingerprints = set(
            spectral["protocol_fingerprint"].dropna().astype(str)
        )
        if spectral_fingerprints != {str(manifest["protocol_fingerprint"])}:
            raise RuntimeError(
                f"{spectral_path} fingerprint mismatch: "
                f"observed={sorted(spectral_fingerprints)}"
            )
        spectral["seed"] = int(seed)
        spectral["optimizer"] = optimizer
        spectral_frames.append(spectral)
    return (
        arm,
        manifests,
        pd.concat(performance_frames, ignore_index=True, sort=False),
        pd.concat(spectral_frames, ignore_index=True, sort=False),
    )


def canonical_columns(performance, spectral):
    performance = performance.copy()
    spectral = spectral.copy()
    for frame in (performance, spectral):
        if "analysis_epoch" in frame and "state_index" not in frame:
            frame["state_index"] = pd.to_numeric(
                frame["analysis_epoch"], errors="coerce"
            )
        if "analysis_state" in frame and "state_index" not in frame:
            frame["state_index"] = pd.to_numeric(
                frame["analysis_state"], errors="coerce"
            )
        if "epoch" in frame and "state_index" not in frame:
            frame["state_index"] = pd.to_numeric(frame["epoch"], errors="coerce")
    if "test_accuracy" not in performance and "test_acc" in performance:
        performance["test_accuracy"] = performance["test_acc"]
    if "layer" not in spectral and "layer_name" in spectral:
        spectral["layer"] = spectral["layer_name"]
    if "fit_variant" not in spectral:
        spectral["fit_variant"] = "unspecified"
    return performance, spectral


## Require all three complete arms


In [ ]:
performance_frames = []
spectral_frames = []
legacy_trace_frames = []
manifests = []
for optimizer_slug in OPTIMIZER_SLUGS:
    arm, arm_manifests, performance, spectral = load_arm_outputs(
        optimizer_slug
    )
    performance, spectral = canonical_columns(performance, spectral)
    performance_frames.append(performance)
    spectral_frames.append(spectral)
    manifests.extend(arm_manifests)
    arm_manifest_by_seed = {
        int(item["seed"]): item for item in arm_manifests
    }
    for seed in SEEDS:
        seed_dir = resolve_seed_dir(optimizer_slug, seed)
        trace_path = require_path(
            seed_dir / "metrics" / "trace_log.csv",
            description="trace-log audit table",
        )
        trace_frame = pd.read_csv(trace_path)
        for column, expected in (
            ("optimizer", optimizer_slug), ("seed", int(seed))
        ):
            if column not in trace_frame:
                raise KeyError(f"{trace_path} lacks identity column {column}")
            observed = set(trace_frame[column].dropna().astype(str))
            if observed != {str(expected)}:
                raise RuntimeError(
                    f"{trace_path} identity mismatch for {column}: "
                    f"observed={sorted(observed)}, expected={expected}"
                )
        if "protocol_fingerprint" not in trace_frame:
            raise KeyError(f"{trace_path} lacks protocol_fingerprint")
        trace_fingerprints = set(
            trace_frame["protocol_fingerprint"].dropna().astype(str)
        )
        expected_fingerprint = str(
            arm_manifest_by_seed[int(seed)]["protocol_fingerprint"]
        )
        if trace_fingerprints != {expected_fingerprint}:
            raise RuntimeError(
                f"{trace_path} fingerprint mismatch: "
                f"observed={sorted(trace_fingerprints)}"
            )
        legacy_trace_frames.append(trace_frame)
performance = pd.concat(performance_frames, ignore_index=True, sort=False)
spectral = pd.concat(spectral_frames, ignore_index=True, sort=False)
legacy_trace = pd.concat(
    legacy_trace_frames, ignore_index=True, sort=False
)
provenance_audit = validate_cross_run_provenance(manifests)
assert set(performance["optimizer"]) == set(OPTIMIZER_SLUGS)
assert set(pd.to_numeric(performance["seed"]).astype(int)) == set(SEEDS)
if "test_monitoring_only" in performance:
    assert performance["test_monitoring_only"].astype(int).eq(1).all()
display(pd.DataFrame(manifests))
display(provenance_audit)
display(performance.tail(18))
display(spectral.tail(24))


## Accuracy and alpha trajectories


In [ ]:
performance_summary, _ = plot_seed_ci(
    performance,
    x="state_index",
    metric="test_accuracy",
    groups=("optimizer",),
    title="Monitoring-only test accuracy",
    ylabel="Test accuracy",
    output_path=OUTPUT_ROOT_PATH / "comparison" / "test_accuracy_95ci.png",
)
primary_all = spectral.copy()
required_fit_columns = {"selection_role", "fit_variant"}
missing_fit_columns = required_fit_columns.difference(primary_all.columns)
if missing_fit_columns:
    raise KeyError(
        "Strict fixed-point comparison requires columns "
        f"{sorted(missing_fit_columns)}"
    )
primary_all = primary_all[
    primary_all["selection_role"].astype(str).isin(
        ["primary", "preregistered_primary"]
    )
    & primary_all["fit_variant"].astype(str).eq("clip_xmax")
].copy()
if primary_all.empty:
    raise RuntimeError(
        "No preregistered primary rows with exact fit_variant=clip_xmax exist"
    )
alpha_plot = primary_all.copy()
if "status" in alpha_plot:
    alpha_plot = alpha_plot[alpha_plot["status"].astype(str).eq("ok")]
alpha_plot = alpha_plot[
    pd.to_numeric(alpha_plot["alpha"], errors="coerce").notna()
]
alpha_summary, _ = plot_seed_ci(
    alpha_plot,
    x="state_index",
    metric="alpha",
    groups=("optimizer", "layer"),
    title="Raw-weight layer alpha: complete-run uncertainty",
    ylabel="Power-law density exponent alpha",
    reference=2.0,
    allow_incomplete=True,
    incomplete_output_path=(
        OUTPUT_ROOT_PATH / "comparison" / "alpha_incomplete_ci_groups.csv"
    ),
    output_path=OUTPUT_ROOT_PATH / "comparison" / "alpha_95ci.png",
)


## Independent-support trace-log audit


In [ ]:
trace_column = next(
    (
        name
        for name in ("trace_log_per_eval", "trace_log_midpoint_per_eval")
        if name in primary_all.columns
    ),
    None,
)
if trace_column is None:
    raise KeyError(
        "Spectral table has no trace-log-per-evaluation column. "
        "Run strict offline spectral analysis first."
    )
if "support_selected_from_same_trace_log" not in primary_all:
    raise KeyError(
        "Trace-log rows must declare support_selected_from_same_trace_log."
    )
same_curve = primary_all["support_selected_from_same_trace_log"]
if same_curve.dtype != bool:
    same_curve = same_curve.astype(str).str.strip().str.lower().isin(
        {"1", "true", "yes"}
    )
independent_all = primary_all[~same_curve].copy()
independent_all["support_selected_from_same_trace_log"] = False
if independent_all.empty:
    raise RuntimeError("No independently supported trace-log rows remain")
trace_plot = independent_all[
    pd.to_numeric(independent_all[trace_column], errors="coerce").notna()
].copy()
if "status" in trace_plot:
    trace_plot = trace_plot[trace_plot["status"].astype(str).eq("ok")]
trace_summary, _ = plot_seed_ci(
    trace_plot,
    x="state_index",
    metric=trace_column,
    groups=("optimizer", "layer"),
    title="Independent-support trace-log",
    ylabel="Trace-log per retained mode",
    reference=0.0,
    allow_incomplete=True,
    incomplete_output_path=(
        OUTPUT_ROOT_PATH / "comparison"
        / "trace_log_incomplete_ci_groups.csv"
    ),
    output_path=OUTPUT_ROOT_PATH / "comparison" / "trace_log_95ci.png",
)


## Legacy raw-midpoint trace-log compatibility audit (non-certifying)


In [ ]:
compatibility_dir = OUTPUT_ROOT_PATH / "comparison"
compatibility_dir.mkdir(parents=True, exist_ok=True)
source_column = next(
    (
        name for name in ("support_rank_source", "support_source")
        if name in legacy_trace.columns
    ),
    None,
)
legacy_trace_metric = next(
    (
        name for name in (
            "trace_log_per_eval", "trace_log_midpoint_per_eval"
        ) if name in legacy_trace.columns
    ),
    None,
)
legacy_raw_midpoint = pd.DataFrame()
if (
    source_column is not None
    and legacy_trace_metric is not None
    and "fit_variant" in legacy_trace
):
    legacy_raw_midpoint = legacy_trace[
        legacy_trace["fit_variant"].astype(str).eq("raw")
        & legacy_trace[source_column].astype(str).eq("weightwatcher_midpoint")
    ].copy()
if legacy_raw_midpoint.empty:
    legacy_compatibility_status = pd.DataFrame([{
        "operator_kind": "legacy_raw_weightwatcher_midpoint_trace_log",
        "map_definition": (
            "historical raw WeightWatcher midpoint support; compatibility "
            "audit only and unavailable in these trace_log.csv artifacts"
        ),
        "available": False,
        "certification_eligible": False,
        "reason": "no exact fit_variant=raw, support_source=weightwatcher_midpoint rows",
    }])
    legacy_summary = pd.DataFrame()
else:
    if legacy_trace_metric != "trace_log_per_eval":
        legacy_raw_midpoint["trace_log_per_eval"] = pd.to_numeric(
            legacy_raw_midpoint[legacy_trace_metric], errors="coerce"
        )
    legacy_raw_midpoint["operator_kind"] = (
        "legacy_raw_weightwatcher_midpoint_trace_log"
    )
    legacy_raw_midpoint["map_definition"] = (
        "historical raw WeightWatcher fit with midpoint-defined trace support; "
        "baseline compatibility audit, not independent-support certification"
    )
    legacy_raw_midpoint["certification_eligible"] = False
    if "state_index" not in legacy_raw_midpoint:
        legacy_raw_midpoint["state_index"] = pd.to_numeric(
            legacy_raw_midpoint.get("epoch"), errors="coerce"
        )
    legacy_summary, _ = plot_seed_ci(
        legacy_raw_midpoint,
        x="state_index",
        metric="trace_log_per_eval",
        groups=("optimizer", "layer"),
        title="Legacy raw WeightWatcher midpoint trace-log (audit only)",
        ylabel="Legacy trace-log per retained mode",
        reference=0.0,
        allow_incomplete=True,
        incomplete_output_path=(
            compatibility_dir / "legacy_raw_midpoint_incomplete_ci_groups.csv"
        ),
        output_path=(
            compatibility_dir / "legacy_raw_midpoint_trace_log_95ci.png"
        ),
    )
    legacy_compatibility_status = pd.DataFrame([{
        "operator_kind": "legacy_raw_weightwatcher_midpoint_trace_log",
        "map_definition": legacy_raw_midpoint["map_definition"].iloc[0],
        "available": True,
        "certification_eligible": False,
        "reason": "historical compatibility observable only",
    }])
legacy_raw_midpoint.to_csv(
    compatibility_dir / "legacy_raw_midpoint_trace_log_rows.csv", index=False
)
legacy_summary.to_csv(
    compatibility_dir / "legacy_raw_midpoint_trace_log_95ci.csv", index=False
)
legacy_compatibility_status.to_csv(
    compatibility_dir / "legacy_raw_midpoint_status.csv", index=False
)
display(legacy_compatibility_status)
if not legacy_summary.empty:
    display(legacy_summary)


## Preregistered late-state qualification table


In [ ]:
# Build the exact preregistered last-five state grid from the
# analysis-state performance table. A missing spectral row stays in
# the grid as NaN and fails; it cannot be replaced by an older fit.
for frame_name, frame in (
    ("performance", performance), ("primary spectral", independent_all)
):
    if "step" not in frame:
        if "global_step" in frame:
            frame["step"] = pd.to_numeric(
                frame["global_step"], errors="raise"
            )
        else:
            raise KeyError(f"{frame_name} table has no step/global_step column")
    frame["step"] = pd.to_numeric(frame["step"], errors="raise").astype(int)
primary_identity = ["optimizer", "seed", "layer", "step"]
duplicated = independent_all.duplicated(primary_identity, keep=False)
if duplicated.any():
    raise RuntimeError(
        "Duplicate preregistered primary rows would corrupt the exact late grid:\n"
        + independent_all.loc[duplicated, primary_identity]
        .sort_values(primary_identity)
        .to_string(index=False)
    )
expected_layers = ("fc1.weight", "fc2.weight", "fc3.weight")
late_grid_rows = []
for optimizer in OPTIMIZER_SLUGS:
    for seed in SEEDS:
        states = performance[
            performance["optimizer"].astype(str).eq(optimizer)
            & pd.to_numeric(performance["seed"], errors="coerce").eq(seed)
        ].sort_values("step")
        if states["step"].duplicated().any():
            raise RuntimeError(
                f"Duplicate performance analysis step for {optimizer}, seed={seed}"
            )
        states = states.tail(5)
        if len(states) != 5:
            raise RuntimeError(
                f"Expected five scheduled late performance states for {optimizer}, "
                f"seed={seed}; found {len(states)}"
            )
        for _, state in states.iterrows():
            for layer in expected_layers:
                late_grid_rows.append({
                    "optimizer": optimizer,
                    "seed": int(seed),
                    "layer": layer,
                    "step": int(state["step"]),
                    "expected_epoch": int(state.get("epoch", state["state_index"])),
                    "expected_state_index": int(state["state_index"]),
                })
late_grid = pd.DataFrame(late_grid_rows)
qualification_input = late_grid.merge(
    independent_all,
    on=primary_identity,
    how="left",
    validate="one_to_one",
    indicator="late_grid_merge_status",
)
if "ks_D" not in qualification_input and "D" in qualification_input:
    qualification_input["ks_D"] = qualification_input["D"]
required_status_columns = {"status", "fit_ok", "trace_log_status"}
missing_status_columns = required_status_columns.difference(
    qualification_input.columns
)
if missing_status_columns:
    raise KeyError(
        "Strict fixed-point qualification requires fit/trace status columns: "
        f"{sorted(missing_status_columns)}"
    )
failed_measurement = (
    qualification_input["late_grid_merge_status"].astype(str).ne("both")
    | qualification_input["status"].astype(str).str.lower().ne("ok")
    | ~boolean_series(qualification_input["fit_ok"])
    | qualification_input["trace_log_status"].astype(str).str.lower().ne("ok")
)
qualification_input["measurement_valid_for_qualification"] = ~failed_measurement
for metric in ("alpha", "ks_D", "n_tail", trace_column):
    qualification_input.loc[failed_measurement, metric] = np.nan
if trace_column != "trace_log_per_eval":
    qualification_input["trace_log_per_eval"] = qualification_input[trace_column]
qualification = reporting.qualify_fixed_point(
    qualification_input,
    alpha_target=2.0,
    alpha_half_width=ALPHA_TOLERANCE,
    max_ks_D=MAXIMUM_KS_D,
    minimum_tail=MINIMUM_TAIL,
    trace_log_tolerance=TRACE_LOG_PER_EVAL_TOLERANCE,
    persistence_measurements=5,
    required_fraction=0.80,
).sort_values(["optimizer", "layer", "seed"])
expected_layers = set(expected_layers)
arm_seed_rows = []
for (optimizer, seed), group in qualification.groupby(
    ["optimizer", "seed"], dropna=False
):
    observed_layers = set(group["layer"].astype(str))
    if observed_layers != expected_layers:
        raise RuntimeError(
            f"Fixed-point verdict requires all layers for {optimizer}, "
            f"seed={seed}; observed={sorted(observed_layers)}"
        )
    arm_seed_rows.append({
        "optimizer": optimizer,
        "seed": int(seed),
        "layers_required": len(expected_layers),
        "layers_qualified": int(group["fixed_point_qualified"].astype(bool).sum()),
        "all_layers_qualified": bool(group["fixed_point_qualified"].astype(bool).all()),
        "fc3_low_rank_warning": True,
    })
arm_seed_verdict = pd.DataFrame(arm_seed_rows)
expected_arm_seed_rows = len(OPTIMIZER_SLUGS) * len(SEEDS)
if len(arm_seed_verdict) != expected_arm_seed_rows:
    raise RuntimeError(
        f"Expected {expected_arm_seed_rows} optimizer/seed spectral verdicts; "
        f"found {len(arm_seed_verdict)}"
    )
for optimizer in OPTIMIZER_SLUGS:
    observed_seeds = set(
        arm_seed_verdict.loc[
            arm_seed_verdict["optimizer"].astype(str).eq(optimizer), "seed"
        ].astype(int)
    )
    if observed_seeds != set(SEEDS):
        raise RuntimeError(
            f"Spectral verdict for {optimizer} has seeds "
            f"{sorted(observed_seeds)}, expected {sorted(SEEDS)}"
        )
optimizer_verdict = (
    arm_seed_verdict.groupby("optimizer", as_index=False)
    .agg(
        seeds_present=("seed", "nunique"),
        seeds_all_layers_qualified=("all_layers_qualified", "sum"),
        all_three_seeds_reproduce=("all_layers_qualified", "all"),
    )
)
if set(optimizer_verdict["optimizer"].astype(str)) != set(OPTIMIZER_SLUGS):
    raise RuntimeError("Optimizer spectral verdict is missing an arm")
if not optimizer_verdict["seeds_present"].eq(len(SEEDS)).all():
    raise RuntimeError("Optimizer verdict is missing a preregistered seed")
baseline_verdict = {
    "require_all_layers": True,
    "require_all_three_seeds": True,
    "all_optimizer_arms_reproduce": bool(
        optimizer_verdict["all_three_seeds_reproduce"].all()
    ),
    "accuracy_role": "descriptive monitoring-only; no qualification threshold",
}
comparison_dir = OUTPUT_ROOT_PATH / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
qualification.to_csv(
    comparison_dir / "late_state_preregistered_qualification.csv",
    index=False,
)
qualification_input.to_csv(
    comparison_dir / "late_state_exact_grid_audit.csv", index=False
)
arm_seed_verdict.to_csv(
    comparison_dir / "arm_seed_all_layers_verdict.csv", index=False
)
optimizer_verdict.to_csv(
    comparison_dir / "optimizer_three_seed_reproducibility.csv", index=False
)
provenance_audit.to_csv(
    comparison_dir / "cross_run_provenance_audit.csv", index=False
)
(comparison_dir / "baseline_verdict.json").write_text(
    json.dumps(baseline_verdict, indent=2) + "\n", encoding="utf-8"
)
performance_summary.to_csv(
    comparison_dir / "performance_summary_95ci.csv", index=False
)
alpha_summary.to_csv(comparison_dir / "alpha_summary_95ci.csv", index=False)
trace_summary.to_csv(
    comparison_dir / "trace_log_summary_95ci.csv", index=False
)
display(qualification)
display(arm_seed_verdict)
display(optimizer_verdict)
display(baseline_verdict)


## Final raw-ESD PDF/CCDF panels with fit-window markers


In [ ]:
panel_dir = comparison_dir / "final_esd_pdf_ccdf"
panel_dir.mkdir(parents=True, exist_ok=True)
if "step" not in primary_all:
    if "global_step" not in primary_all:
        raise KeyError("Primary spectral rows lack step/global_step")
    primary_all["step"] = pd.to_numeric(
        primary_all["global_step"], errors="raise"
    ).astype(int)
final_grid_rows = []
for optimizer in OPTIMIZER_SLUGS:
    for seed in SEEDS:
        seed_dir = resolve_seed_dir(optimizer, seed)
        resolved_payload = json.loads(
            require_path(
                Path(seed_dir) / "resolved_config.json",
                description="resolved training config",
            ).read_text(encoding="utf-8")
        )
        resolved_values = dict(
            resolved_payload.get("config", resolved_payload)
        )
        _, _, completion = validate_run_identity(
            seed_dir, optimizer_slug=optimizer, seed=seed
        )
        expected_epoch = int(resolved_values["epochs"])
        expected_step = int(completion["global_step"])
        refs = tuple(
            list_analysis_checkpoints(Path(seed_dir) / "checkpoints")
        )
        if not refs:
            raise RuntimeError(f"No analysis checkpoints beneath {seed_dir}")
        if (
            int(refs[-1].epoch) != expected_epoch
            or int(refs[-1].global_step) != expected_step
        ):
            raise RuntimeError(
                f"Final immutable checkpoint does not match completion for {seed_dir}"
            )
        exact_performance = performance[
            performance["optimizer"].astype(str).eq(optimizer)
            & pd.to_numeric(performance["seed"], errors="coerce").eq(seed)
            & pd.to_numeric(performance["epoch"], errors="coerce").eq(expected_epoch)
            & pd.to_numeric(performance["step"], errors="coerce").eq(expected_step)
        ]
        if len(exact_performance) != 1:
            raise RuntimeError(
                f"Expected one exact final performance row for {optimizer}, "
                f"seed={seed}, epoch={expected_epoch}, step={expected_step}; "
                f"found {len(exact_performance)}"
            )
        for layer in ("fc1.weight", "fc2.weight", "fc3.weight"):
            final_grid_rows.append({
                "optimizer": optimizer, "seed": int(seed), "layer": layer,
                "epoch": expected_epoch, "step": expected_step,
            })
final_grid = pd.DataFrame(final_grid_rows)
final_rows = final_grid.merge(
    primary_all,
    on=["optimizer", "seed", "layer", "epoch", "step"],
    how="left",
    validate="one_to_one",
    indicator="final_primary_merge_status",
)
missing_final = final_rows[
    final_rows["final_primary_merge_status"].astype(str).ne("both")
]
if not missing_final.empty:
    raise RuntimeError(
        "Missing exact-final preregistered primary spectral rows; earlier "
        "states are never backfilled:\n"
        + missing_final[["optimizer", "seed", "layer", "epoch", "step"]]
        .to_string(index=False)
    )
panel_index = []
for _, row in final_rows.iterrows():
    optimizer = str(row["optimizer"])
    seed = int(row["seed"])
    layer = str(row["layer"])
    step = int(row.get("step", row.get("global_step", row["state_index"])))
    epoch = int(row.get("epoch", row["state_index"]))
    seed_dir = resolve_seed_dir(optimizer, seed)
    esd_path = require_path(
        seed_dir / "metrics" / "esd"
        / f"esd_epoch_{epoch:05d}_step_{step:09d}.npz",
        description="final raw weight ESD archive",
    )
    with np.load(esd_path) as archive:
        if layer not in archive:
            raise KeyError(f"{esd_path} lacks {layer}; keys={archive.files}")
        values = np.asarray(archive[layer], dtype=float)
    safe_layer = layer.replace(".", "_")
    target = panel_dir / f"{optimizer}_seed_{seed}_{safe_layer}.png"
    finger = row.get("num_fingers", row.get("finger_count", "unknown"))
    variant = row.get("fit_variant", "unknown")
    fig = plotting.plot_pdf_ccdf(
        values,
        fit_row=row,
        title=(
            f"{optimizer} | seed={seed} | {layer} | "
            f"variant={variant}, fingers={finger}"
        ),
        output=target,
    )
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    panel_index.append({
        "optimizer": optimizer, "seed": seed, "layer": layer,
        "epoch": epoch, "step": step, "fit_variant": variant,
        "finger_policy": "WeightWatcher fix_fingers=clip_xmax",
        "num_fingers": finger, "xmin": row.get("xmin", np.nan),
        "xmax": row.get("xmax", np.nan), "figure": str(target),
        "fit_status": row.get("status", "unknown"),
        "fit_warning": row.get("warning", ""),
    })
panel_index = pd.DataFrame(panel_index)
expected_panels = len(OPTIMIZER_SLUGS) * len(SEEDS) * 3
if len(panel_index) != expected_panels:
    raise RuntimeError(
        f"Expected {expected_panels} final layer/arm/seed panels; got {len(panel_index)}"
    )
panel_index.to_csv(comparison_dir / "final_esd_panel_index.csv", index=False)
display(panel_index)


A failed row falsifies the strict late-state qualification for that
seed/layer under the declared tolerance; it is not repaired by
pooling layers or selecting a different checkpoint. A passing FC3
row remains weak because FC3 has at most ten positive modes.
